In [ ]:
%pip install -U transformers datasets peft accelerate bitsandbytes

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /kaggle/working/Active-Reading--Pattern-Recognition

os.getcwd()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#test

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from itertools import chain
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# --- Config ---
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
DATA_PATH = "Datasets/simple_wiki_corpus.json"
MAX_SEQ_LENGTH = 2048
LEARNING_RATE = 2e-4

# --- Dataset ---
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load Model (QLoRA) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# --- Prep for k-bit training ---
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- LoRA ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- Tokenization ---
def tokenize_function(examples):
    return tokenizer(examples["text"])

def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    result = {
        k: [t[i:i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2
)

lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    num_proc=2
)

# --- Training Args ---
training_args = TrainingArguments(
    output_dir="./qlora_qwen4b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LEARNING_RATE,
    num_train_epochs=1,
    fp16=True,
    logging_steps=5,
    save_steps=5,
    save_total_limit=2,
    report_to="none",
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    ),
)

print(" Starting QLoRA repetition fine-tuning...")
trainer.train(resume_from_checkpoint=True)

print(" Saving adapter...")
model.save_pretrained("./final_qlora_adapter")
tokenizer.save_pretrained("./final_qlora_adapter")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_PATH = "./final_qlora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
gen_kwargs = dict(
    max_new_tokens=128,
    do_sample=False,
    temperature=0.0,
)

def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

base_only_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_only_model.eval()

prompts = [
    "Who received the IEEE Frank Rosenblatt Award in 2010?",
]

for p in prompts:
    print("=" * 80)
    print("PROMPT:", p)

    print("\nBASE:")
    print(generate(base_only_model, p))

    print("\nADAPTED:")
    print(generate(model, p))